In [1]:
import os
import pandas as pd
import numpy as np

save_dir = "/Users/teraimao/experiment/data/解析結果"

models = {
    "gbsw": {
        "base_csv": "gbsw_common_successID_baselineB_strictPAMPA.csv",
        "shape_csv": "shape_orientation_gbsw.csv",
        "out_csv": "gbsw_common_successID_baselineB_strictPAMPA_shape30.csv",
    },
    "hdgb": {
        "base_csv": "hdgb_common_successID_baselineB_strictPAMPA.csv",
        "shape_csv": "shape_orientation_hdgb.csv",
        "out_csv": "hdgb_common_successID_baselineB_strictPAMPA_shape30.csv",
    },
    "hdgbvdw": {
        "base_csv": "hdgbvdw_common_successID_baselineB_strictPAMPA.csv",
        "shape_csv": "shape_orientation_hdgbvdw.csv",
        "out_csv": "hdgbvdw_common_successID_baselineB_strictPAMPA_shape30.csv",
    },
}

id_col = "ID"
pos_col = "Pos_Num"

shape_cols = [
    "Aspect_Ratio_3D_Avg",
    "Aspect_Ratio_3D_Std",
    "Theta_LongAxis_Z_Avg",
    "Theta_LongAxis_Z_Std",
    "Z_over_XY_Avg",
    "Z_over_XY_Std",
]

created_files = []

for model, cfg in models.items():
    print("\n" + "="*80)
    print(model)

    base_path = os.path.join(save_dir, cfg["base_csv"])
    shape_path = os.path.join(save_dir, cfg["shape_csv"])
    out_path = os.path.join(save_dir, cfg["out_csv"])

    base = pd.read_csv(base_path, low_memory=False)
    shape = pd.read_csv(shape_path, low_memory=False)

    print("base shape:", base.shape)
    print("shape raw:", shape.shape)

    base[id_col] = pd.to_numeric(base[id_col], errors="coerce").astype("Int64")
    shape[id_col] = pd.to_numeric(shape[id_col], errors="coerce").astype("Int64")
    shape[pos_col] = pd.to_numeric(shape[pos_col], errors="coerce").astype("Int64")

    # shape成功行だけ使う
    shape_success = shape[shape["Status"] == "Success"].copy()

    print("shape success rows:", len(shape_success))
    print("shape success unique IDs:", shape_success[id_col].nunique())

    # 必要な6列だけ数値化
    missing = [c for c in shape_cols if c not in shape_success.columns]
    if missing:
        raise ValueError(f"{model}: shape列が足りません: {missing}")

    for c in shape_cols:
        shape_success[c] = pd.to_numeric(shape_success[c], errors="coerce")

    # IDごとにpos1〜5が全部あるものだけ残す
    pos_count = shape_success.groupby(id_col)[pos_col].nunique()
    complete_ids = pos_count[pos_count == 5].index

    shape_complete = shape_success[shape_success[id_col].isin(complete_ids)].copy()

    print("shape complete 5-pos IDs:", len(complete_ids))
    print("shape complete rows:", len(shape_complete))
    print("NaN in shape cols:", shape_complete[shape_cols].isna().sum().sum())

    # 横展開
    wide_parts = []

    for col in shape_cols:
        tmp = shape_complete.pivot(index=id_col, columns=pos_col, values=col)

        # pos1〜pos5がそろっているか確認
        for p in [1, 2, 3, 4, 5]:
            if p not in tmp.columns:
                raise ValueError(f"{model}: {col} の pos{p} がありません")

        tmp = tmp[[1, 2, 3, 4, 5]]
        tmp.columns = [f"{col}_pos{p}" for p in [1, 2, 3, 4, 5]]
        wide_parts.append(tmp)

    shape_wide = pd.concat(wide_parts, axis=1).reset_index()

    shape30_cols = [c for c in shape_wide.columns if c != id_col]

    print("shape_wide shape:", shape_wide.shape)
    print("shape30 cols:", len(shape30_cols))
    print("shape_wide NaN:", shape_wide[shape30_cols].isna().sum().sum())

    # 既存strict PAMPA CSVに結合
    before_rows = len(base)

    merged = base.merge(shape_wide, on=id_col, how="inner")

    print("merged shape:", merged.shape)
    print("base rows before:", before_rows)
    print("rows after inner merge:", len(merged))
    print("dropped IDs from base:", before_rows - len(merged))
    print("merged unique IDs:", merged[id_col].nunique())
    print("duplicate IDs:", merged[id_col].duplicated().sum())
    print("shape30 NaN:", merged[shape30_cols].isna().sum().sum())

    # 保存
    merged.to_csv(out_path, index=False)
    created_files.append(out_path)

    print("saved:", out_path)

print("\n✅ created files")
for f in created_files:
    print(f)


gbsw
base shape: (6441, 167)
shape raw: (42500, 53)
shape success rows: 38205
shape success unique IDs: 7653
shape complete 5-pos IDs: 7615
shape complete rows: 38075
NaN in shape cols: 0
shape_wide shape: (7615, 31)
shape30 cols: 30
shape_wide NaN: 0
merged shape: (6441, 197)
base rows before: 6441
rows after inner merge: 6441
dropped IDs from base: 0
merged unique IDs: 6441
duplicate IDs: 0
shape30 NaN: 0
saved: /Users/teraimao/experiment/data/解析結果/gbsw_common_successID_baselineB_strictPAMPA_shape30.csv

hdgb
base shape: (6441, 167)
shape raw: (42500, 53)
shape success rows: 38240
shape success unique IDs: 7657
shape complete 5-pos IDs: 7630
shape complete rows: 38150
NaN in shape cols: 0
shape_wide shape: (7630, 31)
shape30 cols: 30
shape_wide NaN: 0
merged shape: (6441, 197)
base rows before: 6441
rows after inner merge: 6441
dropped IDs from base: 0
merged unique IDs: 6441
duplicate IDs: 0
shape30 NaN: 0
saved: /Users/teraimao/experiment/data/解析結果/hdgb_common_successID_baselineB_